# Simulator tour
A hands-on tour of the Simulator stage (`galactic_tidal_constraints.simr.simulation`, imported as `sim`): it runs one SPAM galaxy
collision and returns arrays. Run the cells top to bottom.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

In [ ]:
%matplotlib inline
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import simulation as sim

## The 14 collision parameters
A model is 14 numbers. In code it's a plain array of shape `(14,)`, and `sim.PARAM_NAMES` names each
position, using the names in SPAM's Fortran:

| Names | Meaning |
|---|---|
| `x, y, z, vx, vy, vz` | The secondary's position and velocity relative to the primary |
| `mass1, mass2` | The two masses |
| `rout1, rout2` | How far out each disk's particles are placed |
| `phi1, phi2, theta1, theta2` | The two disks' orientations, in degrees |

Galaxy 1 is the primary, and stays at the origin. Galaxy 2 is the secondary.

SPAM itself takes 22 values, usually written as a comma-separated model string. The last 8 (`epsilon1/2`
and six `rscale`) do nothing in the extended potential we run, so `sim` appends fixed values for them
(`sim.UNUSED_PARAMS`) when it calls SPAM. `parse_model_string` reads the first 14 values of any longer string.
This model is SDSS 587722984435351614, from JSPAM's `archive/data.txt`, whose lines have 34 values.

In [ ]:
MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"

params = sim.parse_model_string(MODEL_STRING)
dict(zip(sim.PARAM_NAMES, params.tolist()))

To read or change one value by name, look up its position in `sim.PARAM_INDEX`. Copy the array first, so the
original model stays as it was.

In [ ]:
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
print("mass2:", params[sim.PARAM_INDEX["mass2"]], "->", heavier[sim.PARAM_INDEX["mass2"]])

## How to simulate: `SimulatorConfig`
Everything that sets a run's result, apart from the 14 parameters, is one `sim.SimulatorConfig`:

| Field | Default | Meaning |
|---|---|---|
| `seed` | 0 | Where the disk particles start (see Seeds below) |
| `n1`, `n2` | 1000, 1000 | Particles in the primary's and secondary's disks |
| `heat1`, `heat2` | 0.0, 0.0 | Random motion added to each disk's particle velocities |
| `heat_mode` | `"centered"` | How heat is added (see Disk heat below) |
| `lnl`, `r_scale`, `m_halo`, `m_bulge`, `h_bulge`, `h_disk` | 0.1, 10, 5.8, 0.3333, 0.2, 1.0 (Wallin's, except `lnl`: 0.001) | The unit galaxy's mass model (see below) |
| `normalization` | `"total"` | What the unit galaxy is divided by (see below) |
| `start`, `start_pass`, `no_pass` | `"pass"`, 1, −5.0 | When the run starts (see Orbits and the start time below) |

It can't be changed after it's made (to try a different value, make a changed copy with `replace`), and it
checks its values when it's made. `asdict` gives every field, defaults included.

In [ ]:
config = sim.SimulatorConfig(n1=2000, n2=2000)
print(config)
print(asdict(config))
print(replace(config, heat1=0.5))

## One SPAM run
`sim.simulate_particles(params, config)` runs the collision and returns x, y, z, vx, vy, vz of every particle at the
end, shape `(n1 + n2 + 1, 6)`. The last row is the secondary's center. Runs always use SPAM's extended potential
(halo, disk and bulge).

In [ ]:
N1, N2 = config.n1, config.n2

particles = sim.simulate_particles(params, config)
print("shape:", particles.shape)
print("secondary's center (x, y, z, vx, vy, vz):", particles[-1])

The secondary's center nearly matches `x, y, z, vx, vy, vz` above. The 14 values describe the system *now*,
as observed. SPAM first runs the two centers backward to find a start time before the encounter, places the
disks there, then runs everything forward and stops at about "now" (a step or two past it). So a run ends at
the observed moment, with the tidal features formed.

Split the rows into the two disks and plot x against y:

In [ ]:
primary = particles[:N1]
secondary = particles[N1:N1 + N2]
center2 = particles[-1]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(primary[:, 0], primary[:, 1], s=1, label="primary disk")
ax.scatter(secondary[:, 0], secondary[:, 1], s=1, label="secondary disk")
ax.plot(0, 0, "k+", ms=12, label="primary center")
ax.plot(center2[0], center2[1], "kx", ms=12, label="secondary center")
ax.set_aspect("equal")
ax.set_xlabel("x (SPAM units)")
ax.set_ylabel("y (SPAM units)")
ax.legend(markerscale=5)
plt.show()

## Seeds
`config.seed` (0 by default) sets where the disk particles start. The same seed gives exactly the same particles; a
different seed gives a different, equally valid set. The secondary's center has no random numbers, so it's the same in
every case.

SPAM draws the same random numbers for every model: each particle's ring, radius, angle and heat kicks. So with one
seed, every model's disks share one pattern, stretched by `rout` and tilted by `theta` and `phi`. Score differences
between models then come only from their parameters, never from a lucky draw. Change the seed only to test or show how
much the draw matters.

In [ ]:
same_seed = sim.simulate_particles(params, config)  # seed 0, the default
other_seed = sim.simulate_particles(params, replace(config, seed=1))
print("seed 0 again, particles identical:", np.array_equal(same_seed, particles))
print("seed 1, particles match:          ", np.allclose(other_seed[:-1], particles[:-1]))
print("seed 1, secondary's center matches:", np.allclose(other_seed[-1], particles[-1]))

## The starting state
`sim.initial_state` takes the same arguments as `simulate_particles`, and returns the particles before SPAM takes a step.
With the same config, it's exactly the state `simulate_particles` starts from. The secondary starts far away, before the
encounter.

In [ ]:
start = sim.initial_state(params, config)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(start[:N1, 0], start[:N1, 1], s=1, label="primary disk")
ax.scatter(start[N1:N1 + N2, 0], start[N1:N1 + N2, 1], s=1, label="secondary disk")
ax.set_aspect("equal")
ax.set_title("start")
ax.legend(markerscale=5)
plt.show()

## Unperturbed galaxies
`sim.unperturbed_particles(params, config)` gives the two galaxies as they'd look now if they had never
interacted: the same disks a run starts with (same seed), placed around today's centers with no steps. It takes the
same arguments as `simulate_particles` and returns the same shape, so the two line up with no translating. It skips
the start-time search, so it costs about as much as `initial_state` (a few hundredths of a second).

Positions depend only on the 14 parameters and the seed; heat and the mass model change only the velocities.

In [ ]:
unperturbed = sim.unperturbed_particles(params, config)
print("shape:", unperturbed.shape)
print("secondary's center:", unperturbed[-1], "(exactly the model's x, y, z, vx, vy, vz)")
print("primary's disk unchanged from the start:", np.array_equal(unperturbed[:N1], start[:N1]))

## Disk heat
`heat1` and `heat2` add random motion to each disk's particle velocities (0 by default). `heat_mode` sets how:

| `heat_mode` | Kick added to each of vx, vy, vz | Effect |
|---|---|---|
| `"centered"` (default) | between −heat/2 and +heat/2 | Random motion, no drift |
| `"wallin"` (the original) | between 0 and heat | The same spread, but always positive, so a heated disk also drifts toward +x, +y, +z by about heat/2 |

The kicks show up as the difference between a cold and a hot start with one seed:

In [ ]:
HEAT = 0.5
cold = sim.initial_state(params, config)
for mode in ["centered", "wallin"]:
    hot = sim.initial_state(params, replace(config, heat1=HEAT, heat2=HEAT, heat_mode=mode))
    kick = hot[:-1, 3:] - cold[:-1, 3:]
    print(f"{mode:8s}  kick range: {kick.min():+.3f} to {kick.max():+.3f}   mean per axis: {kick.mean(axis=0).round(3)}")

## The galaxy's mass model
Both galaxies are scaled copies of one **unit galaxy**: a halo, a disk and a bulge, stored as tables over
radius. The force on a particle is the table's force × `mass1` (or `mass2`) from the parameters.
`sim.mass_profile(config)` returns those tables. One SPAM distance unit is about 15 kpc. What each value does,
in the code and physically, is in `docs/spam_physics.md`.

| Field | Default | What it does |
|---|---|---|
| `lnl` | 0.1 (Wallin's: 0.001) | Strength of dynamical friction on the secondary |
| `r_scale` | 10 | The halo's core radius. It also sets the tables' radius grid, so it stretches every component |
| `m_halo`, `m_bulge` | 5.8, 0.3333 | Halo and bulge masses |
| `h_bulge` | 0.2 | Bulge scale length |
| `h_disk` | 1.0 | Disk scale length, which also sets the disk's mass |
| `normalization` | `"total"` | `"total"`: divide by the actual total, so the unit galaxy's mass is 1 and the values above only shape it. `"wallin"` (the original): divide by a fixed 7.13333, so changing a mass changes the total |

### The unit galaxy
Enclosed mass of each component, and the force a particle feels. The halo, disk and bulge tables end at r = 2,
so the mass stops growing there.

In [ ]:
profile = sim.mass_profile(config)
r = profile["radius"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name in ["halo", "disk", "bulge", "total"]:
    axes[0].plot(r, profile[name], label=name)
axes[0].set_xlim(0, 3)
axes[0].set_xlabel("radius (SPAM units)")
axes[0].set_ylabel("enclosed mass (galaxy mass = 1)")
axes[0].legend()
axes[1].plot(r, profile["force"], label="SPAM's force")
axes[1].plot(r[1:], profile["total"][1:] / r[1:] ** 2, "--", label="enclosed mass / r²")
axes[1].set_xlim(0, 10)
axes[1].set_yscale("log")
axes[1].set_xlabel("radius (SPAM units)")
axes[1].legend()
plt.show()
for name in ["halo", "disk", "bulge", "total"]:
    print(f"{name:6s} {profile[name][-1]:.5f}")
print("total with wallin normalization:", sim.mass_profile(replace(config, normalization="wallin"))["total"][-1].round(5))

The dashed line is plain gravity from the enclosed mass. SPAM's force matches it (within 0.2%) from r = 0.05 to
the table's end at r = 10. Inside r = 0.05 it's deliberately smoothed, so particles near the center don't get a
huge kick. Past r = 10 the table ends, and SPAM keeps using its last row, so the force stops falling.

### What each value does
Each row doubles (or halves) one value and shows the total enclosed mass, with `"total"` normalization so every
curve ends at 1. `lnl` isn't in the tables: it only sets dynamical friction (below).

In [ ]:
changes = {
    "m_halo": [2.9, 11.6], "m_bulge": [0.15, 0.6], "h_bulge": [0.1, 0.4], "h_disk": [0.5, 2.0], "r_scale": [5.0, 20.0],
}
base = sim.mass_profile(config)
fig, axes = plt.subplots(len(changes), 1, figsize=(12, 3 * len(changes)), sharex=True, sharey=True)
for ax, (name, values) in zip(axes, changes.items()):
    ax.plot(base["radius"], base["total"], "k", label=f"{getattr(config, name)} (default)")
    for value in values:
        p = sim.mass_profile(replace(config, **{name: value}))
        ax.plot(p["radius"], p["total"], label=str(value))
    ax.set_xlim(0, 2)
    ax.set_ylabel("enclosed mass")
    ax.set_title(name, loc="left")
    ax.legend(loc="lower right")
axes[-1].set_xlabel("radius (SPAM units)")
fig.tight_layout()
plt.show()

**`r_scale` is the odd one.** It's the halo's core radius, but the tables' radius grid is built from it too,
so it stretches every component, while SPAM keeps reading the tables at fixed distances. In force that shows up
as (10 / `r_scale`)² outside the core:

In [ ]:
base_force = sim.mass_profile(config)["force"]
row = np.searchsorted(r, 5.0)  # r = 5
for r_scale in [5.0, 10.0, 20.0]:
    force = sim.mass_profile(replace(config, r_scale=r_scale))["force"]
    print(f"r_scale {r_scale:5.1f}: force at r = 5 is {force[row] / base_force[row]:.3f} x the default"
          f"   (10 / r_scale)^2 = {(10 / r_scale) ** 2:.3f}")

### Effect on a run
Raising `m_halo` to 11.6: with `"wallin"` the galaxies get much heavier, so the encounter changes a lot. With
`"total"` they keep their masses, and only the distribution changes. Same seed in every panel.

In [ ]:
small = replace(config, n1=1500, n2=1500)
runs = {
    "total, defaults": small,
    "total, m_halo 11.6": replace(small, m_halo=11.6),
    "wallin, m_halo 11.6": replace(small, m_halo=11.6, normalization="wallin"),
}
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (title, cfg) in zip(axes, runs.items()):
    final = sim.simulate_particles(params, cfg)
    ax.scatter(final[:cfg.n1, 0], final[:cfg.n1, 1], s=1)
    ax.scatter(final[cfg.n1:-1, 0], final[cfg.n1:-1, 1], s=1)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.show()

### Dynamical friction (`lnl`)
Friction slows the secondary as it moves through the primary's halo, but the halo's density table ends at
r = 2, so it only acts when the secondary passes inside that. The SDSS model above never does. Line 3 of
JSPAM's `data.txt` does. Every run still ends at the observed position, so friction changes the orbit's
*history*: when and how close the pass was, and so when the run starts. `sim` uses 0.1 (Wallin's 0.001 is
nearly none). Without friction the orbit repeats (two equal passes); with it, the older pass disappears.

In [ ]:
close = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                  12.97415, 135.11902, 97.54035, 305.01225])
fig, ax = plt.subplots(figsize=(14, 5))
for lnl in [0.0, 0.001, 0.0025, 0.01,]:
    cfg = replace(config, lnl=lnl)
    passes = sim.find_passes(close, cfg)
    print(f"lnl {lnl:5.3f}: passes at t = {[round(p['t'], 2) for p in passes]}, "
          f"distance {[round(p['distance'], 3) for p in passes]}, run starts {sim.get_timing(close, cfg)['t_start']:.2f}")
    search = sim.search_orbit(close, cfg)
    ax.plot(search[:, 6], np.linalg.norm(search[:, :3], axis=1), label=f"lnl {lnl:g}")
ax.axhline(2, color="0.5", ls=":", label="r = 2: friction acts below this")
ax.set_xlabel("time (SPAM units, now = 0)")
ax.set_ylabel("distance between centers")
ax.legend()
plt.show()

Two runs of the close model that differ only in `lnl`: their particles with each run's orbit, and the distance between
the centers over time. Friction changes the **history** (when the run starts, when and how close the pass was), but
after the pass the secondary is outside r = 2, so no friction acts, and both runs are pinned to the same observed
position and velocity now. Their recent orbits are therefore the same, and the runs look alike near now. The SDSS
model above never comes inside r = 2, so for it `lnl` changes nothing at all, not even the search.

In [ ]:
lnl_runs = {lnl: replace(config, n1=2000, n2=2000, lnl=lnl) for lnl in [0.001, 0.1]}
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
for ax, (lnl, cfg) in zip(axes[:2], lnl_runs.items()):
    p = sim.simulate_particles(close, cfg)
    orbit = sim.simulate_orbit(close, cfg)
    ax.scatter(p[:cfg.n1, 0], p[:cfg.n1, 1], s=1, label="primary disk")
    ax.scatter(p[cfg.n1:-1, 0], p[cfg.n1:-1, 1], s=1, label="secondary disk")
    ax.plot(orbit[:, 0], orbit[:, 1], color="k", lw=1, label="secondary's orbit")
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect("equal")
    ax.set_title(f"lnl {lnl}: starts at t = {orbit[0, 6]:.2f}")
    axes[2].plot(orbit[:, 6], np.linalg.norm(orbit[:, :3], axis=1), label=f"lnl {lnl}")
axes[0].legend(loc="upper left", markerscale=5)
axes[2].axhline(2, color="0.5", ls=":", label="r = 2: friction acts below this")
axes[2].set_xlabel("time (SPAM units, now = 0)")
axes[2].set_ylabel("distance between centers")
axes[2].set_title("each run's orbit")
axes[2].legend()
plt.show()

a, b = (sim.simulate_orbit(close, cfg) for cfg in lnl_runs.values())
n = 3000  # the last 3 time units
print(f"last 3 time units: largest difference between the two orbits {np.abs(a[-n:, :3] - b[-n:, :3]).max():.5f}")

## Orbits and the start time
A run starts from the pair **now** (t = 0) and has to begin before their encounter, so the disks are built
undisturbed. One time unit is about 87 Myr.

- `sim.search_orbit(params, config)`: the secondary's path from now back to t = −30, one row per step (about
  0.06 s). Every run's start lies on it.
- `sim.find_passes(params, config)`: every close pass on that path (a local minimum of the distance between the
  centers), most recent first: pass 1, pass 2, …
- `sim.get_timing(params, config)`: the start a run uses, its step count, and the pass it's based on. Nothing is
  run.
- `sim.simulate_orbit(params, config)`: the secondary's center at every step of the run (x, y, z, vx, vy, vz, t),
  with no particles. Its last row is exactly where `simulate_particles` leaves the secondary.

How the start is chosen is set by `config.start`:

| `start` | The run starts |
|---|---|
| `"pass"` (default) | `10 × rout1 / speed` before pass number `start_pass` (default 1, the most recent); at `no_pass` (−5) if there's no such pass back to −30 |
| `"wallin"` | where SPAM's own search puts it: before the closest pass back to −30, and at −5 if that's earlier than −12 |
| a negative number | exactly then |

Only the mass model and the start settings move the centers; particles, heat and the seed don't. Details:
`docs/spam_physics.md`, "The start time" and "`start`, `start_pass`".

In [ ]:
search = sim.search_orbit(params, config)
passes = sim.find_passes(params, config)
for p in passes:
    print(f"pass {p['pass']}: t = {p['t']:7.3f}   distance {p['distance']:.3f}   speed {p['speed']:.3f}")

timing = sim.get_timing(params, config)
print()
for name, value in timing.items():
    print(f"{name:17s} {value}")

orbit = sim.simulate_orbit(params, config)
print("\norbit shape:", orbit.shape, "  last row == the particle run's secondary:",
      np.array_equal(orbit[-1, :6], particles[-1]))

The search path back to −30 (grey) with each pass numbered, and the run's own orbit from its start (orange):

In [ ]:
fig, (ax_xy, ax_sep) = plt.subplots(1, 2, figsize=(16, 6))
ax_xy.plot(search[:, 0], search[:, 1], color="0.75", label="search path, back to −30")
ax_xy.plot(orbit[:, 0], orbit[:, 1], color="C1", lw=2, label="the run's orbit")
ax_xy.plot(0, 0, "k+", ms=14, label="primary (origin)")
ax_xy.plot(*orbit[0, :2], "go", label=f"start, t = {orbit[0, 6]:.2f}")
ax_xy.plot(*orbit[-1, :2], "k^", label="now")
ax_xy.set_aspect("equal")
ax_xy.set_xlabel("x")
ax_xy.set_ylabel("y")
ax_xy.legend(loc="upper right")

ax_sep.plot(search[:, 6], np.linalg.norm(search[:, :3], axis=1), color="0.75", label="search path")
ax_sep.plot(orbit[:, 6], np.linalg.norm(orbit[:, :3], axis=1), color="C1", lw=2, label="the run's orbit")
ax_sep.axvline(orbit[0, 6], color="g", ls="--", label="start")
for p in passes:
    ax_sep.plot(p["t"], p["distance"], "rs")
    ax_sep.annotate(f"pass {p['pass']}", (p["t"], p["distance"]), textcoords="offset points", xytext=(0, -16),
                    ha="center", color="r")
for p in passes:
    x, y = search[np.argmin(np.abs(search[:, 6] - p["t"])), :2]
    ax_xy.plot(x, y, "rs")
    ax_xy.annotate(str(p["pass"]), (x, y), textcoords="offset points", xytext=(6, 6), color="r")
ax_sep.set_xlabel("time (SPAM units, now = 0)")
ax_sep.set_ylabel("distance between centers")
ax_sep.legend()
plt.show()

## Plots from each step
The plots made while each step was built, collected here. Each subsection runs on its own after the setup cells
at the top (it only needs `params`). Collapse a subsection by clicking the bar left of its heading.

### Step 3 · Seeds
The same seed gives the same particles; a different seed moves each particle but keeps the disk's overall shape
and the orbit.

In [ ]:
N = 1000
seeded = sim.SimulatorConfig(n1=N, n2=N)
a = sim.simulate_particles(params, replace(seeded, seed=1))
b = sim.simulate_particles(params, replace(seeded, seed=1))
c = sim.simulate_particles(params, replace(seeded, seed=2))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for ax, (title, run) in zip(axes, [("seed 1", a), ("seed 2", c), ("seed 1 again", b)]):
    ax.scatter(run[:N, 0], run[:N, 1], s=1)
    ax.scatter(run[N:2 * N, 0], run[N:2 * N, 1], s=1)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.show()

### Step 4a · The starting state and Wallin's heat
The disks at the start of a run and at the end:

In [ ]:
N, SEED = 2000, 1
cold_config = sim.SimulatorConfig(n1=N, n2=N)
start = sim.initial_state(params, replace(cold_config, seed=SEED))
end = sim.simulate_particles(params, replace(cold_config, seed=SEED))

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
for ax, (title, state) in zip(axes, [("start", start), ("end", end)]):
    ax.scatter(state[:N, 0], state[:N, 1], s=1, label="primary disk")
    ax.scatter(state[N:2 * N, 0], state[N:2 * N, 1], s=1, label="secondary disk")
    ax.plot(state[-1, 0], state[-1, 1], "kx", ms=12, label="secondary center")
    ax.set_title(title)
    ax.set_aspect("equal")
axes[0].legend(markerscale=5)
plt.show()

With Wallin's heat (`heat_mode="wallin"`, here a large heat of 5 to make it obvious), every kick is between 0 and
heat, never negative, so each disk gains a net velocity of heat/2 on every axis:

In [ ]:
HEAT = 5
wallin_hot = replace(cold_config, heat1=HEAT, heat2=HEAT, heat_mode="wallin")
kick = sim.initial_state(params, replace(wallin_hot, seed=SEED))[:-1, 3:] - start[:-1, 3:]

fig, ax = plt.subplots(figsize=(8, 4))
for axis, name in enumerate(["vx", "vy", "vz"]):
    ax.hist(kick[:, axis], bins=50, histtype="step", label=name)
ax.axvline(0, color="k", lw=0.8)
ax.axvline(HEAT / 2, color="k", ls="--", lw=0.8, label="mean, heat / 2")
ax.set_xlabel("velocity added by heat")
ax.set_ylabel("particles")
ax.legend()
plt.show()

By the end, the heated primary disk has drifted away from its galaxy's center (the red dot is its mean position;
the cross is the primary's center):

In [ ]:
hot_end = sim.simulate_particles(params, replace(wallin_hot, seed=SEED))

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
for ax, (title, state) in zip(axes, [("heat off", end), (f"Wallin's heat ({HEAT})", hot_end)]):
    ax.scatter(state[:N, 0], state[:N, 1], s=1)
    ax.scatter(state[N:2 * N, 0], state[N:2 * N, 1], s=1)
    ax.plot(0, 0, "k+", ms=14)
    ax.plot(*state[:N, :2].mean(axis=0), "ro", ms=6, label="primary disk's mean position")
    ax.set_title(title)
    ax.set_aspect("equal")
axes[0].legend()
plt.show()

### Step 4b · Centered heat
The same heat in both modes: the spread is identical, but `"centered"` kicks are balanced around 0, while
`"wallin"` kicks are shifted up by heat/2.

In [ ]:
HEAT = 0.5
MODES = ["wallin", "centered"]
kicks = {}
runs = {"no heat": end}
for mode in MODES:
    hot = replace(cold_config, heat1=HEAT, heat2=HEAT, heat_mode=mode)
    kicks[mode] = sim.initial_state(params, replace(hot, seed=SEED))[:-1, 3:] - start[:-1, 3:]
    runs[mode] = sim.simulate_particles(params, replace(hot, seed=SEED))

fig, ax = plt.subplots(figsize=(8, 4))
for mode in MODES:
    ax.hist(kicks[mode].ravel(), bins=60, histtype="step", label=mode)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("velocity added by heat (vx, vy and vz together)")
ax.set_ylabel("particles")
ax.legend()
plt.show()

By the end of the run, only Wallin's heat moves the primary disk's mean position off its center:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (title, state) in zip(axes, runs.items()):
    ax.scatter(state[:N, 0], state[:N, 1], s=1)
    ax.scatter(state[N:2 * N, 0], state[N:2 * N, 1], s=1)
    ax.plot(0, 0, "k+", ms=14)
    ax.plot(*state[:N, :2].mean(axis=0), "ro", ms=6, label="primary disk's mean position")
    ax.set_title(title if title == "no heat" else f"{title}, heat {HEAT}")
    ax.set_aspect("equal")
axes[0].legend()
plt.show()

### Step 5a · The mass model
Step 5a's plots are already in this tour, under **The galaxy's mass model**: the unit galaxy, what each value
does, and the effect on a run.

### Step 6a · Timing and orbits
The orbit and the distance between the centers are under **Orbits and the start time** above. The secondary's
path drawn over a particle run:

In [ ]:
N = 2000
run = sim.simulate_particles(params, sim.SimulatorConfig(n1=N, n2=N))
path = sim.simulate_orbit(params)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(run[:N, 0], run[:N, 1], s=1, label="primary disk")
ax.scatter(run[N:2 * N, 0], run[N:2 * N, 1], s=1, label="secondary disk")
ax.plot(path[:, 0], path[:, 1], "k", lw=1, label="secondary's path")
ax.plot(0, 0, "k+", ms=14)
ax.set_aspect("equal")
ax.legend(markerscale=5)
plt.show()

How the mass model moves **SPAM's own** start (`start="wallin"`). A weaker force (`r_scale` 12, 20) starts
earlier, as intended. A slightly stronger one (`r_scale` 8, 9) makes the search pick an older pass near −20; its
start is then before −12, so the run falls back to −5. `m_halo` (with `"total"`) changes nothing here: the
centers stay outside r = 2. Step 6b's `start="pass"` (the default now) fixes this; see below.

In [ ]:
base = sim.SimulatorConfig(start="wallin")
cases = {"default": base, "m_halo 11.6": replace(base, m_halo=11.6)}
cases.update({f"r_scale {v:g}": replace(base, r_scale=v) for v in [5.0, 8.0, 9.0, 12.0, 20.0]})

print(f"{'':12s} {'t_start':>8s} {'found':>8s} {'t_closest':>9s} {'distance':>8s}  fallback")
for label, cfg in cases.items():
    tm = sim.get_timing(params, cfg)
    print(f"{label:12s} {tm['t_start']:8.2f} {tm['t_start_found']:8.2f} {tm['t_closest']:9.2f} "
          f"{tm['closest_distance']:8.2f}  {tm['start_fallback']}")

fig, ax = plt.subplots(figsize=(12, 5))
for label in ["default", "r_scale 9", "r_scale 12", "r_scale 20"]:
    path = sim.simulate_orbit(params, cases[label])
    ax.plot(path[:, 6], np.linalg.norm(path[:, :3], axis=1), label=label)
ax.axvline(-5, color="0.5", ls="-.", label="−5 fallback")
ax.set_xlabel("time (SPAM units, now = 0)")
ax.set_ylabel("distance between centers")
ax.legend()
plt.show()

### Step 6b · Choosing the start
`r_scale` = 9, where SPAM's search goes wrong. The orbit is bound and repeats: both passes are at the same
distance, and SPAM's "closest in the window" happens to pick the older one, then falls back to −5. The three
starts, on the search path:

In [ ]:
strong = sim.SimulatorConfig(r_scale=9.0)
starts = {
    "pass 1 (default)": strong,
    "pass 2": replace(strong, start_pass=2),
    "SPAM's (wallin)": replace(strong, start="wallin"),
}
search = sim.search_orbit(params, strong)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(search[:, 6], np.linalg.norm(search[:, :3], axis=1), color="0.6", label="search path")
for p in sim.find_passes(params, strong):
    ax.annotate(f"pass {p['pass']}", (p["t"], p["distance"]), textcoords="offset points", xytext=(0, -16),
                ha="center", color="r")
    ax.plot(p["t"], p["distance"], "rs")
for i, (label, cfg) in enumerate(starts.items()):
    tm = sim.get_timing(params, cfg)
    ax.axvline(tm["t_start"], color=f"C{i}", ls="--", label=f"{label}: start {tm['t_start']:.2f}")
ax.set_xlabel("time (SPAM units, now = 0)")
ax.set_ylabel("distance between centers")
ax.legend()
plt.show()

The particle runs. Here SPAM's −5 is still before pass 1 (at −2.87), so its run shows that pass, but with a lead
of about 2.1 instead of 3.8: the disks start closer, more by luck than design. Pass 2's run starts before the
older pass, so the disks go through both:

In [ ]:
N = 1500
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (label, cfg) in zip(axes, starts.items()):
    run = sim.simulate_particles(params, replace(cfg, n1=N, n2=N))
    ax.scatter(run[:N, 0], run[:N, 1], s=1)
    ax.scatter(run[N:2 * N, 0], run[N:2 * N, 1], s=1)
    ax.set_title(f"{label}, start {sim.get_timing(params, cfg)['t_start']:.2f}")
    ax.set_aspect("equal")
plt.show()

### Step 7a · Unperturbed galaxies
The same disks (one seed) with and without the encounter, on the same axes:

In [ ]:
N = 2000
cfg = sim.SimulatorConfig(n1=N, n2=N)
pair = {
    "unperturbed": sim.unperturbed_particles(params, cfg),
    "run (perturbed)": sim.simulate_particles(params, cfg),
}
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)
for ax, (title, state) in zip(axes, pair.items()):
    ax.scatter(state[:N, 0], state[:N, 1], s=1, label="primary disk")
    ax.scatter(state[N:2 * N, 0], state[N:2 * N, 1], s=1, label="secondary disk")
    ax.plot(0, 0, "k+", ms=12)
    ax.plot(state[-1, 0], state[-1, 1], "kx", ms=12)
    ax.set_title(title)
    ax.set_aspect("equal")
axes[0].legend(markerscale=5)
plt.show()

Each galaxy close up, unperturbed in grey and the run in color, centered on its own center:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (name, rows, color) in zip(axes, [("primary", slice(0, N), "C0"), ("secondary", slice(N, 2 * N), "C1")]):
    for state, c, label in [(pair["unperturbed"], "0.6", "unperturbed"), (pair["run (perturbed)"], color, "run")]:
        center = np.zeros(2) if name == "primary" else state[-1, :2]
        ax.scatter(*(state[rows, :2] - center).T, s=1, color=c, label=label)
    ax.set_title(name)
    ax.set_xlim(-12, 12)
    ax.set_ylim(-12, 12)
    ax.set_aspect("equal")
    ax.legend(markerscale=5)
plt.show()